In [ ]:
# Mount Google Drive so this notebook can read/write files stored there
# (our dataset and the saved model both live in Drive, not on the temporary Colab disk)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# --- Configuration / hyperparameters used throughout the notebook ---
IMAGE_SIZE = (64, 64)        # bump to (128, 128) if you want more detail and have the RAM/time to spare
BATCH_SIZE = 32              # number of images fed to the model per training step
EPOCHS = 10                  # max number of passes over the training data (EarlyStopping may stop sooner)

# Path to the dataset inside Google Drive. The nested "asl_alphabet_train" folders reflect how the
# Kaggle dataset unpacks combined with the copytree step below - it really is this deeply nested.
DATA_DIR = "/content/drive/MyDrive/asl_data_new/asl_alphabet_train/asl_alphabet_train/asl_alphabet_train/asl_alphabet_train"
LOCAL_DATA_DIR = "/content/asl_local/asl_alphabet_train"  # fast working copy, lives on the Colab VM's local disk
MAX_IMAGES_PER_CLASS = 1000  # cap per class so we don't need to load all ~87k images per class (keeps runs fast)

In [ ]:
# If the dataset isn't already in Drive, download it from Kaggle and copy it there.
# This way the ~1GB download only ever happens once - later runs just reuse the Drive copy.
import os

if not os.path.isdir(DATA_DIR):
    print("Dataset not found in Drive yet — downloading and copying it there now...")
    import kagglehub
    import shutil
    kaggle_path = kagglehub.dataset_download("grassknoted/asl-alphabet")  # downloads to a local cache dir
    shutil.copytree(kaggle_path, "/content/drive/MyDrive/asl_data_new/asl_alphabet_train/asl_alphabet_train", dirs_exist_ok=True)
    print("Copied to Drive.")
else:
    print("Dataset already present in Drive — skipping download.")

Dataset already present in Drive — skipping download.


In [ ]:
# Install MediaPipe (Google's hand-tracking/landmark library).
# Not used later in this notebook yet - installed for possible future work on
# hand-landmark-based features instead of/alongside raw pixel images.
!pip install mediapipe -q

In [ ]:
# Copy a limited subset of images (up to `max_per_class` per class) from the slow
# Drive-mounted folder over to the Colab VM's local disk. Local disk reads are much
# faster than reading from Drive, which matters a lot once we start training.
# Uses a thread pool so many files copy in parallel instead of one at a time.
import os
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm  # notebook-friendly progress bars

def copy_dataset_subset(src_dir, dst_dir, max_per_class=None, max_workers=16):
    if os.path.isdir(dst_dir):
        shutil.rmtree(dst_dir)  # start clean - wipe any partial/previous copy first

    print("Scanning source directory...")
    copy_jobs = []
    class_names = sorted(os.listdir(src_dir))
    for class_name in tqdm(class_names, desc="Scanning classes"):
        src_class_dir = os.path.join(src_dir, class_name)
        if not os.path.isdir(src_class_dir):
            continue  # skip stray files that aren't class folders
        dst_class_dir = os.path.join(dst_dir, class_name)
        os.makedirs(dst_class_dir, exist_ok=True)

        img_files = os.listdir(src_class_dir)
        if max_per_class is not None:
            img_files = img_files[:max_per_class]  # only take the first N images for this class

        for img_file in img_files:
            copy_jobs.append((
                os.path.join(src_class_dir, img_file),
                os.path.join(dst_class_dir, img_file)
            ))

    print(f"Copying {len(copy_jobs)} files to local disk...")
    # Submit every copy as its own thread-pool job so files copy concurrently
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(shutil.copy2, s, d) for s, d in copy_jobs]
        for _ in tqdm(as_completed(futures), total=len(futures), desc="Copying"):
            pass  # just here to advance the progress bar as each copy finishes

    print(f"Done. Copied {len(copy_jobs)} files.")

# Actually run the copy: DATA_DIR (Drive) -> LOCAL_DATA_DIR (local disk), capped per class
copy_dataset_subset(DATA_DIR, LOCAL_DATA_DIR, max_per_class=MAX_IMAGES_PER_CLASS)

Scanning source directory...


Scanning classes:   0%|          | 0/29 [00:00<?, ?it/s]

Copying 29000 files to local disk...


Copying:   0%|          | 0/29000 [00:00<?, ?it/s]

Done. Copied 29000 files.


### Result: dataset copied locally
All 29 classes were found and **29,000 images total (1,000 per class)** were copied from Drive to the
Colab VM's local disk in about a minute and a half, ready for fast loading in the next step.

In [ ]:
# Core imports for building, training and evaluating the CNN
import tensorflow as tf
import cv2                                                             # image reading/resizing/color conversion
import numpy as np
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator    # for on-the-fly data augmentation
from tensorflow.keras.callbacks import EarlyStopping                   # stops training once val_loss stops improving
from sklearn.model_selection import train_test_split
from matplotlib import pyplot

In [ ]:
# Load every image from disk into memory as a normalized tensor, along with its class label.
# Each subfolder of `data_dir` is treated as one class (e.g. "A", "B", ..., "space").
def load_data(data_dir, max_per_class=None):
    image_data = []
    labels = []
    class_names = []

    for item_name in sorted(os.listdir(data_dir)):
        item_path = os.path.join(data_dir, item_name)
        if not os.path.isdir(item_path):
            continue  # skip anything that isn't a class folder

        class_names.append(item_name)
        idx = len(class_names) - 1  # integer label for this class, based on its position in the sorted list

        img_files = os.listdir(item_path)
        if max_per_class is not None:
            img_files = img_files[:max_per_class]

        for img_file in img_files:
            img_path = os.path.join(item_path, img_file)
            pixels = pyplot.imread(img_path)  # RGB

            # Normalize color channels: some images may be grayscale or have an alpha channel,
            # so convert everything to plain 3-channel RGB for consistency
            if len(pixels.shape) == 2:  # Grayscale
                pixels = cv2.cvtColor(pixels, cv2.COLOR_GRAY2RGB)
            elif pixels.shape[2] == 4:  # RGBA
                pixels = cv2.cvtColor(pixels, cv2.COLOR_RGBA2RGB)

            img_resized = cv2.resize(pixels, IMAGE_SIZE)        # resize to the fixed input size the model expects
            img_array = tf.keras.utils.img_to_array(img_resized)
            image_data.append(img_array)
            labels.append(idx)

    image_data = tf.convert_to_tensor(image_data) / 255.0  # scale pixel values from [0, 255] down to [0, 1]
    labels = tf.convert_to_tensor(labels)
    return image_data, labels, class_names

print("Loading data...")
image_data, labels, class_names = load_data(LOCAL_DATA_DIR, max_per_class=MAX_IMAGES_PER_CLASS)
print(f"Classes ({len(class_names)}): {class_names}")
print(f"Image data shape: {image_data.shape}")

Loading data...
Classes (29): ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']
Image data shape: (29000, 64, 64, 3)


### Result: images loaded into memory
Data loading picked up all **29 classes** (`A`–`Z`, plus `del`, `nothing`, `space`) and produced an image
tensor of shape **(29000, 64, 64, 3)** — 29,000 images, each resized to 64×64 pixels with 3 color channels.

In [ ]:
# Convert from TensorFlow tensors to plain NumPy arrays, then split into training and
# validation sets (80% train / 20% validation). random_state=42 makes the split reproducible.
image_data_np = image_data.numpy()
labels_np = labels.numpy()

X_train, X_val, y_train, y_val = train_test_split(
    image_data_np, labels_np, test_size=0.2, random_state=42
)
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")

X_train shape: (23200, 64, 64, 3)
X_val shape: (5800, 64, 64, 3)


### Result: train/validation split
The 29,000 images were split 80/20 into **23,200 training images** and **5,800 validation images**,
which the model will never train on directly and will instead be used to check how well it generalizes.

In [ ]:
# Define the CNN architecture: 3 convolution+pooling blocks (32 -> 64 -> 128 filters) that
# progressively extract more abstract visual features, followed by a dense classifier head.
# Dropout layers are sprinkled throughout to reduce overfitting.
def create_model(num_classes):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),                                   # flatten the 2D feature maps into a 1D vector
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),                                 # heavier dropout right before the output layer
        layers.Dense(num_classes, activation='softmax')      # one output probability per ASL class
    ])
    return model

model = create_model(len(class_names))
# sparse_categorical_crossentropy works directly with integer labels (no need to one-hot encode them)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 29)             │         3,741 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 686,941 (2.62 MB)

 Trainable params: 686,941 (2.62 MB)

 Non-trainable params: 0 (0.00 B)

### Result: model architecture
The model summary reports **686,941 trainable parameters (~2.62 MB)** and no non-trainable parameters,
confirming the 3 convolution blocks + dense head defined above compiled successfully and are all being trained.

In [ ]:
# Set up on-the-fly data augmentation: randomly rotate/shift/shear/zoom the training images
# each epoch so the model sees more variety and generalizes better instead of memorizing the
# exact training images.
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=False,  # OFF: a flipped ASL hand sign can mean a different letter, unlike a face/mask
    fill_mode='nearest'
)
datagen.fit(X_train)

# Stop training automatically if validation loss hasn't improved for 5 epochs in a row,
# and roll back the model's weights to whichever epoch had the best validation loss
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("Training...")
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),  # feed in augmented training batches
    validation_data=(X_val, y_val),                          # validation data is left un-augmented
    epochs=EPOCHS,
    callbacks=[early_stopping]
)

val_loss, val_accuracy = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {val_accuracy:.2f}")

Training...
Epoch 1/10
725/725 ━━━━━━━━━━━━━━━━━━━━ 46s 53ms/step - accuracy: 0.1302 - loss: 3.0027 - val_accuracy: 0.3626 - val_loss: 2.0893
Epoch 2/10
725/725 ━━━━━━━━━━━━━━━━━━━━ 35s 48ms/step - accuracy: 0.3452 - loss: 2.0972 - val_accuracy: 0.6916 - val_loss: 1.1176
Epoch 3/10
725/725 ━━━━━━━━━━━━━━━━━━━━ 36s 50ms/step - accuracy: 0.4620 - loss: 1.6637 - val_accuracy: 0.7529 - val_loss: 0.8525
Epoch 4/10
725/725 ━━━━━━━━━━━━━━━━━━━━ 34s 47ms/step - accuracy: 0.5351 - loss: 1.4207 - val_accuracy: 0.8029 - val_loss: 0.6515
Epoch 5/10
725/725 ━━━━━━━━━━━━━━━━━━━━ 35s 48ms/step - accuracy: 0.5874 - loss: 1.2470 - val_accuracy: 0.8431 - val_loss: 0.4916
Epoch 6/10
725/725 ━━━━━━━━━━━━━━━━━━━━ 36s 49ms/step - accuracy: 0.6351 - loss: 1.0990 - val_accuracy: 0.8714 - val_loss: 0.3896
Epoch 7/10
725/725 ━━━━━━━━━━━━━━━━━━━━ 34s 47ms/step - accuracy: 0.6689 - loss: 0.9941 - val_accuracy: 0.8872 - val_loss: 0.3396
Epoch 8/10
725/725 ━━━━━━━━━━━━━━━━━━━━ 41s 47ms/step - accuracy: 0.7000 - los

### Result: training run
Training ran for all **10 epochs** (early stopping never triggered, since validation loss kept improving
throughout). Over the course of training:
- Training accuracy rose from **13.0% → 74.8%**, with training loss falling from **3.00 → 0.75**
- Validation accuracy rose from **36.3% → 93.1%**, with validation loss falling from **2.09 → 0.21**
- Final validation accuracy after evaluation: **0.93 (93%)**

Notice that validation accuracy ends up *higher* than training accuracy. This is expected here because
the training images are being randomly rotated/shifted/zoomed each epoch (making them harder to classify),
while the validation images are evaluated in their original, un-augmented form. The steadily falling
validation loss (no sign of it turning back up) suggests the model was not obviously overfitting by epoch 10.

In [ ]:
# Save the trained model to Google Drive so it persists after this Colab session ends
save_dir = "/content/drive/MyDrive/asl_data_new"
os.makedirs(save_dir, exist_ok=True)
model.save(os.path.join(save_dir, "asl_classifier_model.h5"))  # legacy HDF5 format (see warning in the output below)
print("Model saved.")

Model saved.


### Result: model saved
The model was saved successfully to Drive as `asl_classifier_model.h5`. Keras raised a warning that the
`.h5` format is legacy and recommended switching to the native `.keras` format for future runs.

In [ ]:
# Sanity-check the trained model by predicting on a few individual images (indices 0, 11
# and 9999) and comparing the predicted class against the true (actual) label
for i in [0, 11, 9999]:
    image_array = np.expand_dims(image_data_np[i], axis=0)  # add a batch dimension of size 1
    prediction = model.predict(image_array)
    predicted_class = class_names[np.argmax(prediction)]     # pick the class with the highest predicted probability

    pyplot.imshow(image_data_np[i])
    pyplot.title(f"Predicted: {predicted_class}  |  Actual: {class_names[labels_np[i]]}")
    pyplot.show()

NameError: name 'np' is not defined